# qdmpy Data Exploration

**Target user**: "I want to play around with the data."

Covers: loading raw data, the processing pipeline, spectrum inspection,
fitting parameter maps, B111 xarray access, and iteration (refit with
different parameters).

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import qdmpy
from qdmpy.testing import make_synthetic_odmr_data, make_synthetic_fit_result
from qdmpy import ODMR, ODMRData, NormalizationProcessor, BinningProcessor, OutlierProcessor

## 1. Loading and inspecting raw data

With real data:
```python
from qdmpy import MatlabLoader, ODMRData
loader = MatlabLoader('/data/FOV18x')
raw = ODMRData.from_loader(loader)
```

Here we use synthetic data with the same structure:

In [ ]:
raw = make_synthetic_odmr_data(shape=(32, 32), n_freq=50)

print('Data shape:', dict(zip(raw.data.dims, raw.data.shape)))
print('Polarities:', raw.data.coords['polarity'].values.tolist())
print('Freq ranges:', raw.data.coords['freq_range'].values.tolist())
print('Spatial dims:', raw.scan_dimensions)

# Frequency axes (GHz)
freq_ghz = raw.data.coords['freq_ghz'].values
for i, label in enumerate(['low', 'high']):
    f = freq_ghz[i]
    print(f'  {label}: {f[0]:.4f} – {f[-1]:.4f} GHz  ({len(f)} steps)')

## 2. Inspect a single spectrum

In [ ]:
odmr = ODMR(raw)

# spectrum() returns (freq_GHz, intensity) for one pixel
# Use processed=False to inspect the raw spectrum before any processing
freq, spec = odmr.spectrum(y=8, x=8, polarity='neg', freq_range='low', processed=False)
print(f'Spectrum shape: freq={freq.shape}, intensity={spec.shape}')

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(freq, spec, 'o-', ms=3)
ax.set_xlabel('Frequency (GHz)')
ax.set_ylabel('Fluorescence')
ax.set_title('Single pixel spectrum — raw (pol=neg, range=low)')
plt.tight_layout()
plt.savefig('spectrum_raw.png', dpi=80)
plt.show()

## 3. Processing pipeline

Processors are chained in an `ODMRProcessorManager`.
Each processor returns a **new** `ODMRData` — no in-place mutation.

In [ ]:
odmr.processor_manager.add_processor(NormalizationProcessor())
odmr.process_data()

print('Processors applied:')
for p in odmr.processor_manager.processors:
    print(' ', p.describe())

# Compare raw vs processed at the same pixel
freq_raw, spec_raw = odmr.spectrum(y=8, x=8, polarity='neg', freq_range='low', processed=False)
freq_proc, spec_proc = odmr.spectrum(y=8, x=8, polarity='neg', freq_range='low', processed=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 3), sharey=False)
axes[0].plot(freq_raw, spec_raw, 'o-', ms=3, color='C0', label='raw')
axes[0].set_title('Raw spectrum')
axes[1].plot(freq_proc, spec_proc, 'o-', ms=3, color='C1', label='processed')
axes[1].set_title('Processed (normalised)')
for ax in axes:
    ax.set_xlabel('Frequency (GHz)')
    ax.set_ylabel('Fluorescence')
plt.tight_layout()
plt.savefig('spectrum_compare.png', dpi=80)
plt.show()

## 4. All spectra at one pixel — 2×2 grid

`plot_spectra(y, x)` shows all (polarity × freq_range) combinations.

In [ ]:
odmr.plot_spectra(y=8, x=8)

## 5. Fit result parameter maps

With GPU fitting:
```python
from qdmpy import FitManager
fm = FitManager('ESR14N')
fit_result = fm.fit(odmr.processed_data.data, odmr.processed_data.frequencies, pixel_spacing=4e-6)
```

Here we use a pre-built synthetic FitResult:

In [ ]:
fit_result = make_synthetic_fit_result(shape=(32, 32))
print(fit_result)

In [ ]:
# For multi-range models (ESR14N), parameters have shape (n_pol, n_frange, n_pixels).
# Average across pol/frange to get a spatial map for visualisation.
H, W = fit_result.scan_dimensions

center_map = fit_result.parameters['center'].mean(axis=(0, 1)).reshape(H, W)
chi2_map   = fit_result.parameters['chi2'].mean(axis=(0, 1)).reshape(H, W)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
im0 = axes[0].imshow(center_map, cmap='viridis')
axes[0].set_title('Center frequency (GHz, averaged)')
plt.colorbar(im0, ax=axes[0], shrink=0.8)

im1 = axes[1].imshow(chi2_map, cmap='hot_r')
axes[1].set_title('χ² (fit quality, averaged)')
plt.colorbar(im1, ax=axes[1], shrink=0.8)

plt.tight_layout()
plt.savefig('parameter_maps.png', dpi=80)
plt.show()

## 6. B111 xarray Dataset

`FitResult.b111` returns an `xr.Dataset` with variables
'remanent' and 'induced', each a DataArray with dims (y, x).

In [ ]:
b111 = fit_result.b111
print(b111)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, var in zip(axes, ['remanent', 'induced']):
    data = b111[var].values
    vmax = np.percentile(np.abs(data), 98)
    im = ax.imshow(data, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    ax.set_title(f'B111 {var} (µT)')
    plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.savefig('b111_exploration.png', dpi=80)
plt.show()

## 7. Iteration — change bin factor and refit

Processing the same raw data with different binning gives different
spatial resolution vs SNR trade-offs.

In [ ]:
from qdmpy import BinningProcessor

# Start fresh from original raw data
raw_large = make_synthetic_odmr_data(shape=(32, 32))
odmr2 = ODMR(raw_large)

odmr2.processor_manager.add_processor(BinningProcessor(bin_factor=2))
odmr2.processor_manager.add_processor(NormalizationProcessor())
odmr2.process_data()

proc = odmr2.processed_data
print(f'Raw spatial dims:       {raw_large.scan_dimensions}')
print(f'Binned (2×2) spatial dims: {proc.scan_dimensions}')

## 8. Export to NetCDF and reload

In [ ]:
import tempfile, pathlib
from qdmpy import QDMResult

qdm = QDMResult(fit_result=fit_result)

with tempfile.TemporaryDirectory() as tmp:
    path = pathlib.Path(tmp) / 'my_result.npz'
    qdm.save(path)
    reloaded = QDMResult.load(path)

print('Reload OK:', np.allclose(qdm.b111_remanent, reloaded.b111_remanent))